### Only run this cell when latex backend is available

In [ ]:
import matplotlib.pyplot as plt
plt.style.use("utils/RESS.mplstyle")

# Figure 1 

Demo of the mono-level approach 
Create a 2d plot of the mono-level GP and a 1d plot of the design space quantile estimation

In [ ]:
import pickle 
data_file = "_experiments/RESS/1d_demo/ro_data.pkl"

loaded_objects = []
with open(f'{data_file}', 'rb') as f:
    try:
        while True:
            loaded_objects.append(pickle.load(f))
    except EOFError:
        pass

ro_data, model, gp_model, args = loaded_objects

In [ ]:
import numpy as np
import torch
from utils.datastructures import Grid
import matplotlib.pyplot as plt
import torch.distributions as dist
from scipy.interpolate import interp1d
from scipy import stats
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

ny = 1000
nx = 1000
my_grid = Grid(np.linspace(0.0, 1.0, nx), np.linspace(0.0, 1.0, ny))
XX = my_grid.meshgrid_mat[0]
YY = my_grid.meshgrid_mat[1]

def plot_local_gp(x, gp_model, ax, alpha = 0.5):
    n_xi = 1000

    xi = torch.linspace(0.0, 1.0, n_xi)
    x = torch.ones(xi.shape[0])*x

    x_xi = torch.concatenate((x.unsqueeze(-1), xi.unsqueeze(-1)), axis = 1)

    # Inference
    with torch.no_grad():
        post = gp_model.posterior(x_xi)
        mean = post.mean
        lower, upper = post.confidence_region()
        std = (upper - lower)/2

    # Plot the functions
    ax.plot(x, xi, lower, color='lightgreen', alpha=alpha)
    ax.plot(x, xi, upper, color='lightgreen', alpha=alpha)
    ax.plot(x, xi, mean[:,0], color='black', linewidth = 1.0)

    
    # Create the filled area
    verts = []
    for i in range(len(x) - 1):
        verts.append([
            (x[i], xi[i], lower[i]),
            (x[i+1], xi[i+1], lower[i+1]),
            (x[i+1], xi[i+1], upper[i+1]),
            (x[i], xi[i], upper[i])
        ])

    poly = Poly3DCollection(verts, facecolors='lightgreen', alpha=alpha, edgecolor='none')
    ax.add_collection3d(poly)

    return ax

def plot_sample_quant(x, gp_model, ax, output = True):

    n_xi = 1000
    xi = torch.linspace(0.0, 1.0, n_xi)
    x_vec = torch.ones(xi.shape[0])*x
    x_xi = torch.concatenate((x_vec.unsqueeze(-1), xi.unsqueeze(-1)), axis = 1)

    # Inference
    with torch.no_grad():
        post = gp_model.posterior(x_xi)
        mean = post.mean
        lower, upper = post.confidence_region()
        std = (upper - lower)/2
        gp_sample = post.sample(torch.Size([1]))[:,:,0]

    n_samples = 150
    lb = 0.1
    ub = 0.9
    beta_dist = dist.Beta(4.0, 4.0)
    samples = beta_dist.sample((n_samples,))
    xi_sample = lb + (ub - lb) * samples

    linear_interp = interp1d(xi.numpy(), gp_sample.numpy(), kind='linear', fill_value='extrapolate')
    y = linear_interp(xi_sample.numpy())

    z_lim = ax.get_zlim()
    y_lim = ax.get_ylim()
    x_lim = ax.get_xlim()
    kde = stats.gaussian_kde(y)
    n_kde = 1000
    y_values = np.linspace(z_lim[0], z_lim[1], n_kde)
    kde_values = kde(y_values)

    # Distribution 
    xi_values = torch.linspace(lb, ub, n_kde)
    pdf_values = beta_dist.log_prob((xi_values - lb) / (ub - lb)).exp() / (ub - lb)
    xi_values_np = xi_values.numpy()
    pdf_values_np = pdf_values.numpy()

    # Get the quantile
    y_quant = np.quantile(y, 0.9)

    # Plot input and output sample
    ax.plot(np.ones(n_kde)*x_lim[1], xi_values_np, pdf_values_np/np.max(pdf_values_np) * 0.3 + z_lim[0] , color = "dimgrey")
    ax.scatter(torch.ones(n_samples), samples, z_lim[0]*torch.ones(n_samples), alpha = 0.1, s = 5, color = "black")
    
    if output:
        ax.plot(x_vec+0.0, xi, gp_sample[0,:], color='dimgrey', linestyle = "--", linewidth = 1.0, zorder = 100, label=r'$\mathcal{M}(\cdot \mid \theta)$')
        ax.scatter(torch.ones(n_samples), y_lim[1]*torch.ones(n_samples), y, alpha = 0.1, s = 5, color = "black")
        ax.plot(np.ones(n_kde)*x_lim[1], 1.2 - kde_values/np.max(kde_values) * 0.2, y_values, color = "dimgrey")
        # ax.plot(x_lim[1], y_lim[1], y_quant, marker = "+", color = "red", zorder = 100, label=r'$\tilde{q}_{\tau}^{n,(j)}$')
        ax.plot([x_lim[1]], [y_lim[1]], [y_quant], marker="+", zorder=100, color="red",label=r'$\tilde{q}_{\tau}^{n,(j)}$')
    ax.set_zlim(z_lim)

    return ax


def plot_global_gp(gp_model, x_local = None, ax = None, plot_quant = False, plot_output_sample = False, infill_data = None, z_lim = None):
    ny = 1000
    nx = 1000
    my_grid = Grid(np.linspace(0.0, 1.0, nx), np.linspace(0.0, 1.0, ny))
    grid_nodes = torch.from_numpy(my_grid.nodes)

    # Inference
    with torch.no_grad():
        post = gp_model.posterior(grid_nodes)
        mean = post.mean
        lower, upper = post.confidence_region()
        std = (upper - lower)/2

    lower_grid = lower.reshape(ny, nx)
    upper_grid = upper.reshape(ny, nx)
    mean_grid = mean.reshape(ny, nx)

    # Initialize the plot
    if ax is None:
        figsize = np.array([12,12])/2.54
        fig = plt.figure(figsize=figsize)
        ax = fig.add_subplot(111, projection='3d')

    # Plot the third surface (z3)
    surf3 = ax.plot_surface(XX, YY, lower_grid, alpha=0.5, linewidth=0, antialiased=True, color='lightblue', label = r"$\sigma_{\mathcal{D}}$")
    surf5 = ax.plot_surface(XX, YY, upper_grid, alpha=0.5, linewidth=0, antialiased=True, color='lightblue')
    surf4 = ax.plot_surface(XX, YY, mean_grid, alpha=0.3, linewidth=0, antialiased=False, color='grey', label = r"$\mu_{\mathcal{D}}$")

    if infill_data is not None: ax.scatter(infill_data[:,0], infill_data[:,1], infill_data[:,2], marker = "+", color = "k", zorder = 100, s = 35)

    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([-0.05, 1.2])
    if z_lim is None:
        z_lim_current = ax.get_zlim()
        ax.set_zlim([z_lim_current[0] - 0.1, z_lim_current[1]])
    
    if x_local is not None: ax = plot_local_gp(x = x_local, gp_model= gp_model, ax = ax, alpha = 0.9)
    if plot_quant: ax = plot_sample_quant(x = x_local, gp_model= gp_model, ax = ax, output = plot_output_sample)

    if z_lim is not None:
        ax.set_zlim([z_lim[0], z_lim[1]])
    
    ax.tick_params(labelbottom = True, labelleft = True)
    ax.set_xlabel("$x$", labelpad=-10)
    ax.set_ylabel("$\\xi$", labelpad=-10, rotation = 90.0)
    ax.set_zlabel("$y$", labelpad=-12)
    ax.set_box_aspect(aspect=None, zoom=0.85)

    ax.xaxis.pane.set_facecolor('white')
    ax.yaxis.pane.set_facecolor('white')
    ax.zaxis.pane.set_facecolor('white')
    ax.set_xticks([0.0, 0.25, 0.5, 0.75, 0.95, 1.0])
    ax.set_xticklabels(['$0$', '', '', '','', '$1$'])
    ax.set_yticks([0.0, 0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['$0$', '', '', '', '$1$'])
    ax.set_zticks([0.0, 0.25, 0.5, 0.75, 1.0])
    ax.set_zticklabels(['$0$','', '', '', '$1$'])
    ax.grid(alpha = 0.3)
    ax.tick_params(axis='x', pad=-5)
    ax.tick_params(axis='y', pad=-5)
    ax.tick_params(axis='z', pad=-2)
    ax.view_init(elev=35, azim=315)


    x_fig = 0.45  
    y_fig = 0.095  
        
    ax.text2D(x_fig, y_fig, '$x^c$', 
                  transform=ax.transAxes, 
                  fontsize=9, 
                  ha='center', 
                  va='center')
    

    return ax

import matplotlib.pyplot as plt
figsize = np.array([525,300])/72.27
fig = plt.figure(figsize=figsize)
ax1 = fig.add_subplot(121, projection='3d')
ax1 = plot_global_gp(gp_model, x_local = 0.95, plot_quant= True, plot_output_sample= True, ax = ax1)

handles, labels = ax1.get_legend_handles_labels()


from matplotlib.lines import Line2D
from matplotlib.patches import Patch

custom_handles = [
    Patch(
        facecolor='lightblue',
        edgecolor='none',
        alpha=0.5,
        label=r'$\sigma_{\mathcal{D}}$'
    ),
    Patch(
        facecolor='grey',
        edgecolor='none',
        alpha=0.3,
        label=r'$\mu_{\mathcal{D}}$'
    ),
    Line2D(
        [0], [0],
        color='dimgrey',
        linestyle='--',
        linewidth=1.0,
        label=r'$\mathcal{M}(\cdot \mid \theta)$'
    ),
    Line2D(
        [0], [0],
        color='red',
        marker='+',
        linestyle='None',
        markersize=5,
        markeredgewidth=0.8,
        label=r'$\tilde{q}_{\tau}^{n,(j)}$'
    )
]

# Put legend above the figure
fig.legend(
    custom_handles,
    labels,
    loc='upper center',
    bbox_to_anchor=(0.3, 0.82),
    ncol=3,
    frameon=False,
    columnspacing=0.85
)

fig.tight_layout()
plot_folder = "_plots/RESS"
file_name = "monolevel_level_gp"
fig.savefig(f'{plot_folder}/{file_name}.png', format = 'png', bbox_inches = 'tight', pad_inches = 0.0, dpi = 400)

In [ ]:
from utils.plotting import add_DOE_plot
import seaborn as sns
import matplotlib.pyplot as plt
ccycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

data = ro_data.doe["plot_data"][0]

ref_plot = torch.load("_experiments/RESS/1d_demo/demo_1d.pt", weights_only=False)
ref_domain, ref_q_vec = ref_plot['domain_plot']["x"], ref_plot['domain_plot']["q"]
ref_x_vec = ref_domain.nodes

upper_bound = data["J_mean_vec"] + data["J_std_vec"]
lower_bound = data["J_mean_vec"] - data["J_std_vec"]

figsize = np.array([200,120])/72.27
fig, ax = plt.subplots(figsize = figsize)
h1 = ax.plot(ref_x_vec, ref_q_vec, label = f"q_{args.tau}", color = "dimgray", zorder = 1)
h3 = ax.fill_between(data["x_domain"].nodes[:,0], upper_bound, lower_bound, color=ccycle[0], alpha=0.3, zorder = 50)
h2 = ax.plot(data["x_domain"].nodes[:,0], data["J_mean_vec"], color=ccycle[0], zorder = 60)
sns.despine(ax=ax, offset=0)
ax.set_xlabel("$x$", labelpad=-5)
ax.set_ylabel("$y$", rotation=0, labelpad=-5)
ax.grid(alpha = 0.3)
ax.set_yticks([0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
ax.set_yticklabels(['$0.5$', '', '', '', '', '$1$'])
ax.set_xticks([0.0, 0.25, 0.5, 0.75, 1.0])
ax.set_xticklabels(['$0$', '', '', '', '$1$'])
# ax.tick_params(axis='y', rotation=90) 

labels = ["$q_{0.8}$", "$\mu_{\\widetilde{Q}_{\\tau}}$", "$\sigma_{\\widetilde{Q}_{\\tau}}$"]
handles = [h1[0], h2[0], h3]
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.54, 1.1), ncol=3, frameon = False)

fig.tight_layout()
plot_folder = "_plots/RESS"
file_name = "quant_est"
fig.savefig(f'{plot_folder}/{file_name}.png', format = 'png', bbox_inches = 'tight', pad_inches = 0.0, dpi = 400)